# Gold Aggregation
Test the Gold layer aggregations independently.
Note: You should have run silver_transform first so there is pending Silver data.

In [1]:
from datetime import datetime, timezone
import polars as pl
from electricity_maps.layers.gold import aggregate_gold_mix, aggregate_gold_flows
from electricity_maps.utils.state import PipelineState
from electricity_maps.utils.storage import delta_storage_options
from electricity_maps.config import get_settings

In [2]:
get_settings.cache_clear()
settings = get_settings()
state = PipelineState(settings.data_dir)

now = datetime.now(timezone.utc)
gold_ts = int(now.timestamp())

In [3]:
ts_list = state.pickup_ready("silver")
if not ts_list:
    print("No pending silver data found. Run 03_silver_transform first.")
else:
    print(f"Picked up Silver batches: {ts_list}")
    
    aggregate_gold_mix()
    aggregate_gold_flows()
    
    # Mark state
    state.mark_complete("silver", ts_list)
    state.init_layer("gold", gold_ts, now)
    state.mark_ready("gold", gold_ts, now, len(ts_list))
    
    print("Gold aggregation completed!")

2026-04-25T08:39:21.693176Z [info     ] el_state: no ready rows to pick up [electricity_maps.utils.state] process=silver
No pending silver data found. Run 03_silver_transform first.


In [4]:
# Inspect Gold Mix Summary
print("Gold Daily Mix Summary:")
try:
    path = str(settings.gold_dir / "daily_mix_summary")
    df_gold_mix = pl.read_delta(path, storage_options=delta_storage_options(path))
    display(df_gold_mix.head())
except Exception as e:
    print("Error reading gold mix:", e)

Gold Daily Mix Summary:


zone,date,source,generation_mwh,hours_covered,percentage,zone_name,year,month
str,date,str,f64,i32,f64,str,i32,i32
"""FR""",2026-04-25,"""gas""",2735.859,8,0.633495,"""France""",2026,4
"""FR""",2026-04-24,"""biomass""",18928.921,16,1.948415,"""France""",2026,4
"""FR""",2026-04-24,"""oil""",583.846,16,0.060097,"""France""",2026,4
"""FR""",2026-04-24,"""nuclear""",682672.079,16,70.269641,"""France""",2026,4
"""FR""",2026-04-25,"""coal""",0.0,8,0.0,"""France""",2026,4


In [5]:
# Inspect Gold Daily Imports
print("\nGold Daily Imports:")
try:
    path = str(settings.gold_dir / "daily_imports")
    df_imports = pl.read_delta(path, storage_options=delta_storage_options(path))
    display(df_imports.head())
except Exception as e:
    print("Error reading gold imports:", e)


Gold Daily Imports:


zone,date,source_zone,import_mwh,hours_covered,zone_name,year,month
str,date,str,f64,i32,str,i32,i32
"""FR""",2026-04-24,"""ES""",36529.0,16,"""France""",2026,4
"""FR""",2026-04-24,"""DE""",26120.0,16,"""France""",2026,4
"""FR""",2026-04-25,"""CH""",12511.0,8,"""France""",2026,4
"""FR""",2026-04-24,"""CH""",17895.0,16,"""France""",2026,4
"""FR""",2026-04-24,"""BE""",44133.0,16,"""France""",2026,4
